In [1]:
import pandas as pd
from tqdm.auto import tqdm
import sys
sys.path.append("../")
tqdm.pandas()


In [2]:
from src.repo_utils import clone_and_extract_tree

In [3]:
CONTEXT_WINDOW = 395000 # gpt 5.1's context window is 400k tokens

In [4]:
df_repos = pd.read_csv("../data/8_repo_assessment/full_dataset_LLM_pred.csv")

In [5]:
df_repos["LLM_url"].value_counts()

LLM_url
Appendix                                                                                            88
https://github.com/rvandewater/CASS-PROPEL                                                           2
https://github.com/ohdsi-studies/Covid19PredictionStudies                                            2
https://github.com/pennsignals/eol-onc                                                               2
https://github.com/KCLMMAG/LaiaHumbertVidan                                                          2
                                                                                                    ..
https://github.com/WenjuanW/Risk_Prediction_of_Post-Stroke_30-day_Mortality_SSNAP_and_RiksStroke     1
https://github.com/dgbrow02/esbl_prediction_code                                                     1
https://git.rwth-aachen.de/workgroup-lara/multiclass_segmentation_pancreatic_cancer                  1
https://github.com/emilyliangmd/eipm                             

In [6]:
# subset rows where LLM_url is present and not the literal "Appendix" (case-insensitive)
df_repos_assessment = df_repos[
    df_repos["LLM_url"].notna() &
    (df_repos["LLM_url"].astype(str).str.strip().str.lower() != "appendix")
].copy()

In [7]:
def wrapper_clone_and_extract_tree(repo_url):
    result = clone_and_extract_tree(
        repo_url=repo_url,
        context_window=CONTEXT_WINDOW,
        clone_dir="../data/cloned_repos",
    )
    return {
        "repo_content": result.output,
        "repo_status": result.status,
    }

df_repos_assessment[["repo_content", "repo_status"]] = (
    df_repos_assessment["LLM_url"]
    .progress_apply(wrapper_clone_and_extract_tree)
    .apply(pd.Series)
)

  0%|          | 0/421 [00:00<?, ?it/s]

NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
NumberObject(b'-3317-') invalid; use 0 instead
incorrect startxref pointer(1)


In [8]:
from src.repo_utils import RepoStatus

In [9]:
len(df_repos_assessment[df_repos_assessment["repo_status"] == RepoStatus.OK])

352

In [10]:
len(df_repos_assessment[df_repos_assessment["repo_status"] == RepoStatus.EMPTY])

5

In [11]:
len(df_repos_assessment[df_repos_assessment["repo_status"] == RepoStatus.INACCESSIBLE])

43

In [14]:
len(df_repos_assessment[df_repos_assessment["repo_status"] == RepoStatus.NOT_SUPPORTED])

21

In [16]:
df_repos_assessment.to_csv("../data/8_repo_assessment/full_dataset_with_repo_content.csv", index=False)
df_repos_assessment[df_repos_assessment["repo_status"] == RepoStatus.OK].to_csv("../data/8_repo_assessment/full_dataset_with_repo_content_ok.csv", index=False)

In [15]:
for idx, row in df_repos_assessment[df_repos_assessment["repo_content"].isna()].iterrows():
    print(row['LLM_url'])

https://gite.lirmm.fr/advanse/myoctus
https://www.synapse.org/#!Synapse:syn58983698
https://ehealthinformation.ca/blog/EHIL%20Blog/R%20Files
https://doi.org/10.3389/fphar.2023.1334439
https://github.com/detection-mammography
https://www.rama.mahidol.ac.th/ceb/codes
https://github.com/SydneyBioX/P3_model
https://www.synapse.org/#!Synapse:syn51702348
https://github.com/pqstri/SSc
https://github.com/410312774
https://doi.org/10.11588/data/50WFVL
https://doi.org/10.24433/CO.9261428.v1
https://medical-physics-usz.github.io/
https://github.com/dixiong777/COH_SF_IRTML
https://github.com/UEA-Cancer-Genetics-Lab/ExoGrail_paper
https://picudatalab.com/aberration-detection-model/
https://osf.io/6y8fs/
https://gitlab.developers.cam.ac.uk/stroke/papers/rui-dementia-prediction-in-svd-with-ml
https://data.unityimaging.net/
https://sdr.stanford.edu
https://doi.org/10.24433/CO.4899453.v1
https://huggingface.co/spaces/MSHS-Neurosurgery-Research/NSQIP-PCF
https://github.com/KCLMMAG/LaiaHumbertVidan
https

In [9]:
df_figshare = df_repos_assessment[df_repos_assessment['LLM_url'].str.contains('figshare', case=False, na=False)]
df_figshare

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,full_text,LLM_paper_assessment,LLM_reason,LLM_url,repo_content
45,30870451,The development and internal validation of a m...,"de Graaf MW, Reininga IHF, Heineman E, El Moum...",PLoS One. 2019 Mar 14;14(3):e0213510. doi: 10....,de Graaf MW,PLoS One,2019,2019/03/15,PMC6417777,NaN,10.1371/journal.pone.0213510,==== Front\nPLoS OnePLoS ONEplosplosonePLoS ON...,True,Yes. The study develops and internally validat...,https://doi.org/10.6084/m9.figshare.6945728,"Repository tree\n{\n ""url"": ""https://doi.org/..."
253,37008262,WHO cardiovascular disease risk prediction mod...,"Yang S, Ding Y, Yu C, Guo Y, Pang Y, Sun D, Pe...",Bull World Health Organ. 2023 Apr 1;101(4):238...,Yang S,Bull World Health Organ,2023,2023/04/03,PMC10042093,NaN,10.2471/BLT.22.288645,==== Front\nBull World Health Organ\nBull Worl...,True,Yes. The study performs an external validation...,https://doi.org/10.6084/m9.figshare.21688340,"Repository tree\n{\n ""url"": ""https://doi.org/..."
326,36747065,Towards artificial intelligence-based learning...,"Sun W, Kalmady SV, Sepehrvand N, Salimi A, Nad...",NPJ Digit Med. 2023 Feb 6;6(1):21. doi: 10.103...,Sun W,NPJ Digit Med,2023,2023/02/06,PMC9902450,NaN,10.1038/s41746-023-00765-3,==== Front\nNPJ Digit Med\nNPJ Digit Med\nNPJ ...,True,Yes. The study develops and validates multivar...,https://doi.org/10.6084/m9.figshare.21612786.v1,NaN
327,38762623,Development and validation of machine learning...,"Kalmady SV, Salimi A, Sun W, Sepehrvand N, Nad...",NPJ Digit Med. 2024 May 18;7(1):133. doi: 10.1...,Kalmady SV,NPJ Digit Med,2024,2024/05/18,PMC11102430,NaN,10.1038/s41746-024-01130-8,==== Front\nNPJ Digit Med\nNPJ Digit Med\nNPJ ...,True,Yes. The paper develops and validates multivar...,https://figshare.com/s/b593e8d7bfe7cd8500b1,NaN
355,34420921,Using a Convolutional Neural Network to Predic...,"Cao Y, Näslund I, Näslund E, Ottosson J, Montg...",JMIR Med Inform. 2021 Aug 19;9(8):e25612. doi:...,Cao Y,JMIR Med Inform,2021,2021/08/23,PMC8414302,NaN,10.2196/25612,==== Front\nJMIR Med Inform\nJMIR Med Inform\n...,True,Yes. The study develops and validates a multiv...,https://doi.org/10.6084/m9.figshare.13078943,"Repository tree\n{\n ""url"": ""https://doi.org/..."


In [13]:
print(df_figshare['repo_content'].iloc[1])

Repository tree
{
  "url": "https://doi.org/10.6084/m9.figshare.21688340",
  "tree": [
    {
      "name": "Appendix 1.xlsx",
      "children": null
    }
  ]
}
Files content



In [20]:
len(df_repos_assessment[df_repos_assessment["repo_content"].isna()])

58

In [18]:
github_na_count = df_repos_assessment[
    df_repos_assessment['LLM_url'].str.contains('github', case=False, na=False) & 
    df_repos_assessment['repo_content'].isna()
].shape[0]

print(f"Number of rows with 'github' in LLM_url and NA repo_content: {github_na_count}")

Number of rows with 'github' in LLM_url and NA repo_content: 26


In [21]:
df_repos_assessment.to_csv("../data/8_repo_assessment/full_dataset_with_repo_content.csv", index=False)

In [8]:
df_repos_assessment = pd.read_csv("../data/8_repo_assessment/full_dataset_with_repo_content.csv")

In [17]:
df_repos_assessment[
    df_repos_assessment['LLM_url'].str.contains('osf.io', case=False, na=False) & 
    df_repos_assessment['repo_content'].notna()
]

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,full_text,LLM_paper_assessment,LLM_reason,LLM_url,repo_content
382,38049919,Replicability and reproducibility of predictiv...,"Nickson D, Singmann H, Meyer C, Toro C, Walase...",Diagn Progn Res. 2023 Dec 5;7(1):25. doi: 10.1...,Nickson D,Diagn Progn Res,2023,2023/12/04,PMC10696659,NaN,10.1186/s41512-023-00160-2,==== Front\nDiagn Progn Res\nDiagn Progn Res\n...,True,Yes. The study develops and validates multivar...,https://osf.io/573uw/,"Repository tree\n{\n ""url"": ""https://osf.io/5..."
930,38645766,An evaluation of synthetic data augmentation f...,"Juwara L, El-Hussuna A, El Emam K.",Patterns (N Y). 2024 Feb 29;5(4):100946. doi: ...,Juwara L,Patterns (N Y),2024,2024/04/22,PMC11026977,NaN,10.1016/j.patter.2024.100946,==== Front\nPatterns (N Y)\nPatterns (N Y)\nPa...,True,The study trains and evaluates multivariable l...,https://doi.org/10.17605/OSF.IO/RKF9T,"Repository tree\n{\n ""url"": ""https://doi.org/..."
2481,39741130,Virtual reality-assisted prediction of adult A...,"Wiebe A, Selaskowski B, Paskin M, Asché L, Pak...",Transl Psychiatry. 2024 Dec 31;14(1):508. doi:...,Wiebe A,Transl Psychiatry,2024,2024/12/31,PMC11688437,NaN,10.1038/s41398-024-03217-y,==== Front\nTransl Psychiatry\nTransl Psychiat...,True,Yes. The study develops and validates a multiv...,https://doi.org/10.17605/OSF.IO/EWKGV,"Repository tree\n{\n ""url"": ""https://doi.org/..."
2820,30978258,Development and validation of a clinical model...,"Donovan BM, Breheny PJ, Robinson JG, Baer RJ, ...",PLoS One. 2019 Apr 12;14(4):e0215173. doi: 10....,Donovan BM,PLoS One,2019,2019/04/13,PMC6461273,NaN,10.1371/journal.pone.0215173,==== Front\nPLoS OnePLoS ONEplosplosonePLoS ON...,True,Yes. The study develops and validates a multiv...,https://osf.io/w7aes/,"Repository tree\n{\n ""url"": ""https://osf.io/w..."
3659,36071509,Development of a prognostic model of COVID-19 ...,"Eythorsson E, Bjarnadottir V, Runolfsdottir HL...",Diagn Progn Res. 2022 Sep 8;6(1):17. doi: 10.1...,Eythorsson E,Diagn Progn Res,2022,2022/09/07,PMC9451645,NaN,10.1186/s41512-022-00130-0,==== Front\nDiagn Progn Res\nDiagn Progn Res\n...,True,The study develops a multivariable prognostic ...,https://osf.io/t2bp8/,"Repository tree\n{\n ""url"": ""https://osf.io/t..."
3788,39695442,Comparison between clinician and machine learn...,"Pontén M, Flygare O, Bellander M, Karemyr M, N...",BMC Psychiatry. 2024 Dec 18;24(1):904. doi: 10...,Pontén M,BMC Psychiatry,2024,2024/12/18,PMC11653784,NaN,10.1186/s12888-024-06391-x,==== Front\nBMC Psychiatry\nBMC Psychiatry\nBM...,True,Yes. The study develops and evaluates a multiv...,https://osf.io/vym96/,"Repository tree\n{\n ""url"": ""https://osf.io/v..."
4451,35804071,Is the performance at the implicit association...,"Epifania OM, Robusto E, Anselmi P.",Psychol Res. 2023 Apr;87(3):737-750. doi: 10.1...,Epifania OM,Psychol Res,2023,2022/07/08,PMC10017590,NaN,10.1007/s00426-022-01703-w,==== Front\nPsychol Res\nPsychol Res\nPsycholo...,True,Yes. The study fits the criteria because it de...,https://osf.io/y2qak/,"Repository tree\n{\n ""url"": ""https://osf.io/y..."
5041,39487383,A Novel Method to Predict Carbohydrate and Ene...,"Rothschild JA, Hofmeyr S, McLaren SJ, Maunder E.",Sports Med. 2025 Mar;55(3):753-774. doi: 10.10...,Rothschild JA,Sports Med,2025,2024/11/02,PMC11985602,NaN,10.1007/s40279-024-02131-z,==== Front\nSports Med\nSports Med\nSports Med...,True,The study develops multivariable prediction mo...,https://osf.io/nqsu7/,"Repository tree\n{\n ""url"": ""https://osf.io/n..."
5237,39856750,Words to live by: Using medic impressions to i...,"Weidman AC, Sedor-Schiffhauer Z, Zikmund C, Sa...",Acad Emerg Med. 2025 May;32(5):516-525. doi: 1...,Weidman AC,Acad Emerg Med,2025,2025/01/25,PMC12077067,NIHMS2042371,10.1111/acem.15067,==== Front\nAcad Emerg Med\nAcad Emerg Med\n10...,True,Yes. The study develops and validates multivar...,https://osf.io/pjfzw/,"Reposit

In [10]:
zenodo_urls = df_repos_assessment[df_repos_assessment['LLM_url'].str.contains('zenodo', case=False, na=False)]['LLM_url']
print(zenodo_urls)

20        https://doi.org/10.5281/zenodo.8125003
354       https://doi.org/10.5281/zenodo.5078131
1569    http://dx.doi.org/10.5281/zenodo.1471616
1650      https://doi.org/10.5281/zenodo.6759730
1768      https://doi.org/10.5281/zenodo.6956451
1859      https://doi.org/10.5281/zenodo.7888547
1986      https://doi.org/10.5281/zenodo.5074026
2643      https://doi.org/10.5281/zenodo.5627167
3058           https://zenodo.org/record/7817247
3153      https://doi.org/10.5281/zenodo.4560078
3564      https://doi.org/10.5281/zenodo.3893846
4271           https://zenodo.org/record/7330268
4330      https://doi.org/10.5281/zenodo.4836276
4370      https://doi.org/10.5281/zenodo.6107815
4919      https://doi.org/10.5281/zenodo.4077342
Name: LLM_url, dtype: object
